# Tutorial 2: Finite-Horizon Navigation on Non-Convex Glyphs (`'G'`)

This tutorial reproduces the standardized finite-horizon glyph benchmark in Appendix A / Figure 3. It does not establish a non-equilibrium steady state (NESS).

### Why Non-Convex Letterforms?
The upright sans-serif letter `'G'` contains an inward-pointing spur and a C-shaped cavity. In the reported run, Standard Langevin visits 33.2% of glyph pixels and has 33.7% terminal coverage; A-Block visits 97.8% and has 99.2% terminal coverage (`output/typography_metrics.json`). These outcomes are specific to this synthetic potential and evaluation protocol, not a general geometric trapping guarantee.

The Clifford rotor term $\sigma I \nabla V(\mathbf{x})$ adds a tangential drift component and the dilator term $-\kappa \nabla V(\mathbf{x})$ changes the gradient-directed drift. The learned state-dependent coefficients are not constrained to satisfy the stationarity conditions in Proposition 1.

In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from reproduce_typography_figure3 import build_glyph_potential, simulate_typography_ensemble
from src.metrics import calc_glyph_metrics

In [ ]:
mask, V, grad_x, grad_y, r_dish = build_glyph_potential(size=400, char='G')
print(f'Constructed 400x400 glyph potential | Circular aperture radius r_dish = {r_dish} px')

configs = [
    ('Brownian Dynamics (f = 0)',        0.0,  0.0, 'brownian'),
    ('Standard Langevin (f = -grad V)',  0.0,  1.0, 'langevin'),
    ('Rotor-Only (kappa = 0)',          11.0,  0.0, 'rotor'),
    ('A-Block Full Clifford (Ours)',    11.0,  1.8, 'ablock'),
]

print(f"{'Method':<34} | {'Precision (%)':<15} | {'Traj Visitation (%)':<20} | {'Terminal Cov (%)'}")
print('-' * 92)
for name, sigma, kappa, mode in configs:
    pts, traj, visited_grid = simulate_typography_ensemble(
        grad_x, grad_y, mask, r_dish=r_dish, N=1200, T=4000, dt=2.0,
        sigma=sigma, kappa=kappa, noise_amp=0.85, seed=42, mode=mode
    )
    prec, term_cov, visit_cov = calc_glyph_metrics(pts, mask, visited_grid=visited_grid, radius=8)
    print(f"{name:<34} | {prec:<14.1f}% | {visit_cov:<19.1f}% | {term_cov:.1f}%")